# Cross-validation for time series

Which rows go into train and test in each fold, printed on 8 rows so you can see it.

**What's in here**
- `KFold` (and why `shuffle=True` interleaves)
- `TimeSeriesSplit`: fold boundaries, `gap`, `test_size`
- walk-forward by hand: expanding and rolling windows
- `cross_val_score` / `cross_validate` and the `neg_` sign convention
- `GridSearchCV` with a time-series splitter; reading `cv_results_`
- the gap / embargo when the target is h steps ahead
- learning-curve table
- on the real data: shuffled vs time-ordered scores, walk-forward, alpha search
- checklist: is my evaluation honest?

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

## 1. KFold on 8 rows

Eight rows indexed 0..7. `KFold(n_splits=4)` without shuffling takes consecutive blocks.

In [2]:
from sklearn.model_selection import KFold

X8 = pd.DataFrame({"x": range(8)})
for fold, (tr, te) in enumerate(KFold(n_splits=4).split(X8)):
    print(f"fold {fold}: train {list(tr)}  test {list(te)}")

fold 0: train [2, 3, 4, 5, 6, 7]  test [0, 1]
fold 1: train [0, 1, 4, 5, 6, 7]  test [2, 3]
fold 2: train [0, 1, 2, 3, 6, 7]  test [4, 5]
fold 3: train [0, 1, 2, 3, 4, 5]  test [6, 7]


With `shuffle=True` the test rows are scattered and every test row has training neighbours on both sides.

In [3]:
for fold, (tr, te) in enumerate(KFold(n_splits=4, shuffle=True, random_state=0).split(X8)):
    print(f"fold {fold}: train {list(tr)}  test {list(te)}")

fold 0: train [0, 1, 3, 4, 5, 7]  test [2, 6]
fold 1: train [0, 2, 3, 4, 5, 6]  test [1, 7]
fold 2: train [1, 2, 4, 5, 6, 7]  test [0, 3]
fold 3: train [0, 1, 2, 3, 6, 7]  test [4, 5]


Even without shuffle, fold 0 trains on rows 2..7 and tests on rows 0,1: it uses the future
to predict the past. Neither is a forecast.

## 2. TimeSeriesSplit

Training always precedes testing, and the training set grows.

In [4]:
from sklearn.model_selection import TimeSeriesSplit

for fold, (tr, te) in enumerate(TimeSeriesSplit(n_splits=3).split(X8)):
    print(f"fold {fold}: train {list(tr)}  test {list(te)}")

fold 0: train [0, 1]  test [2, 3]
fold 1: train [0, 1, 2, 3]  test [4, 5]
fold 2: train [0, 1, 2, 3, 4, 5]  test [6, 7]


`gap=1` leaves one row out between train and test in every fold.

In [5]:
for fold, (tr, te) in enumerate(TimeSeriesSplit(n_splits=3, gap=1).split(X8)):
    print(f"fold {fold}: train {list(tr)}  test {list(te)}")

fold 0: train [0]  test [2, 3]
fold 1: train [0, 1, 2]  test [4, 5]
fold 2: train [0, 1, 2, 3, 4]  test [6, 7]


Why a gap: if the target at row t is "the value at t+1" (`shift(-1)`), then the last
training row's target *is* the first test row's value. Here it is on 6 rows.

In [6]:
s = pd.DataFrame({"value": [10, 11, 12, 13, 14, 15]})
s["target_next"] = s["value"].shift(-1)
s

,value,target_next
0,10,11.0
1,11,12.0
2,12,13.0
3,13,14.0
4,14,15.0
5,15,NaN


Train on rows 0..2, test on rows 3..5: row 2's target (13) equals row 3's value, which is a
test-set fact. With `gap=1` row 2 is dropped from training. For an h-step target use `gap=h`.

`test_size` fixes the length of each test block.

In [7]:
for fold, (tr, te) in enumerate(TimeSeriesSplit(n_splits=2, test_size=2).split(X8)):
    print(f"fold {fold}: train {list(tr)}  test {list(te)}")

fold 0: train [0, 1, 2, 3]  test [4, 5]
fold 1: train [0, 1, 2, 3, 4, 5]  test [6, 7]


## 3. Walk-forward by hand

Nine rows, refit every 3 rows. Expanding window: train on everything before the block.

In [8]:
X9 = pd.DataFrame({"x": range(9)})
for start in [3, 6]:
    train = X9.iloc[:start]
    test = X9.iloc[start:start + 3]
    print(f"train rows {list(train.index)}  ->  test rows {list(test.index)}")

train rows [0, 1, 2]  ->  test rows [3, 4, 5]
train rows [0, 1, 2, 3, 4, 5]  ->  test rows [6, 7, 8]


Rolling window: keep only the last 3 rows for training (adapts faster, less data).

In [9]:
for start in [3, 6]:
    train = X9.iloc[start - 3:start]
    test = X9.iloc[start:start + 3]
    print(f"train rows {list(train.index)}  ->  test rows {list(test.index)}")

train rows [0, 1, 2]  ->  test rows [3, 4, 5]
train rows [3, 4, 5]  ->  test rows [6, 7, 8]


In each fold you fit, predict the test block, and keep the predictions. Concatenating them
gives one out-of-sample series to score.

In [10]:
from sklearn.linear_model import LinearRegression

y9 = pd.Series([1, 2, 3, 4, 5, 6, 7, 8, 9]) * 2 + 1
preds = []
for start in [3, 6]:
    m = LinearRegression().fit(X9.iloc[:start], y9.iloc[:start])
    preds.append(pd.Series(m.predict(X9.iloc[start:start + 3]), index=X9.index[start:start + 3]))
oos = pd.concat(preds)
pd.DataFrame({"y": y9, "oos_pred": oos})

,y,oos_pred
0,3,NaN
1,5,NaN
2,7,NaN
3,9,9.0
4,11,11.0
5,13,13.0
6,15,15.0
7,17,17.0
8,19,19.0


Rows 0..2 have no prediction: they were never in a test block.

## 4. cross_val_score and the sign convention

sklearn maximises scores, so error metrics are returned **negative**: `neg_root_mean_squared_error`.

In [11]:
from sklearn.model_selection import cross_val_score, cross_validate

X8 = pd.DataFrame({"x": range(8)})
y8 = pd.Series([3, 5, 7, 9, 11, 13, 15, 17]) + pd.Series([0, 1, 0, -1, 0, 1, 0, -1])
scores = cross_val_score(LinearRegression(), X8, y8, cv=TimeSeriesSplit(n_splits=3), scoring="neg_root_mean_squared_error")
print("per fold  :", scores.round(3))
print("RMSE      :", (-scores).round(3), " mean", round(-scores.mean(), 3))

per fold  : [-3.162 -1.838 -0.935]
RMSE      : [3.162 1.838 0.935]  mean 1.979


`cross_validate` returns several metrics at once plus timings.

In [12]:
cvres = cross_validate(LinearRegression(), X8, y8, cv=TimeSeriesSplit(n_splits=3),
                       scoring=["neg_root_mean_squared_error", "r2"], return_train_score=True)
pd.DataFrame(cvres).round(3)

,fit_time,score_time,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error,test_r2,train_r2
0,0.002,0.002,-3.162,-0.000,-39.000,1.000
1,0.003,0.003,-1.838,-0.548,-0.502,0.914
2,0.002,0.002,-0.935,-0.685,-2.498,0.962


## 5. GridSearchCV with a time-series splitter

Two alphas for Ridge inside a pipeline. The parameter name is `step__param`.

In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

pipe = Pipeline([("scale", StandardScaler()), ("ridge", Ridge())])
gs = GridSearchCV(pipe, param_grid={"ridge__alpha": [0.1, 10.0]},
                  cv=TimeSeriesSplit(n_splits=3), scoring="neg_root_mean_squared_error")
gs.fit(X8, y8)
print("best_params_:", gs.best_params_)
pd.DataFrame(gs.cv_results_)[["param_ridge__alpha", "split0_test_score", "split1_test_score", "split2_test_score", "mean_test_score", "rank_test_score"]].round(3)

best_params_: {'ridge__alpha': 0.1}


,param_ridge__alpha,split0_test_score,split1_test_score,split2_test_score,mean_test_score,rank_test_score
0,0.1,-2.869,-1.954,-0.817,-1.880,1
1,10.0,-2.016,-5.284,-4.292,-3.864,2


`mean_test_score` is negative RMSE; the least negative wins (rank 1).

**Pitfall:** the score you report after choosing alpha on these folds is optimistic; you
picked the best of several. Confirm on a later hold-out period, or nest the search inside an
outer time-series split.

## 6. Learning-curve table

Fit on the first n rows, test on the last 2. Noisy at such tiny sizes, but 2 rows is
hopeless and 5-6 rows are fine; on real data the curve flattens once the model has enough.

In [14]:
rows = []
for n in [2, 3, 4, 5, 6]:
    m = LinearRegression().fit(X8.iloc[:n], y8.iloc[:n])
    err = np.sqrt(((m.predict(X8.iloc[6:]) - y8.iloc[6:]) ** 2).mean())
    rows.append({"train_rows": n, "test_rmse": round(err, 3)})
pd.DataFrame(rows)

,train_rows,test_rmse
0,2,7.071
1,3,0.972
2,4,1.530
3,5,0.566
4,6,0.935


## 7. On the real data

Lagged features for a 1-hour-ahead consumption model. Compare shuffled KFold with TimeSeriesSplit.

In [15]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).sort_values("time").reset_index(drop=True)
df["lag1"] = df["consumption_mwh"].shift(1)
df["lag24"] = df["consumption_mwh"].shift(24)
df["lag168"] = df["consumption_mwh"].shift(168)
df["hour"] = df["time"].dt.hour
df = df.dropna().reset_index(drop=True)
feats = ["lag1", "lag24", "lag168", "temp_c", "hour"]
X, y = df[feats], df["consumption_mwh"]
print(X.shape)
X.head(3)

(17352, 5)


,lag1,lag24,lag168,temp_c,hour
0,31353.0,28476.8,26858.4,-1.13,0
1,28210.5,28227.5,26177.8,-1.67,1
2,26205.3,28457.9,26229.4,-1.96,2


In [16]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=20, max_depth=8, random_state=0)
kf = cross_val_score(rf, X, y, cv=KFold(n_splits=5, shuffle=True, random_state=0), scoring="r2")
ts = cross_val_score(rf, X, y, cv=TimeSeriesSplit(n_splits=5), scoring="r2")
print("shuffled KFold R2   :", kf.round(3), " mean", round(kf.mean(), 3))
print("TimeSeriesSplit R2  :", ts.round(3), " mean", round(ts.mean(), 3))

shuffled KFold R2   : [0.974 0.972 0.972 0.973 0.973]  mean 0.973
TimeSeriesSplit R2  : [0.962 0.96  0.963 0.975 0.968]  mean 0.966


The shuffled score is higher: neighbours leak. The gap is small here (a smooth series, a
shallow forest) and grows with more flexible models; report the time-ordered one.

Walk-forward with monthly refits, expanding window, Ridge.

In [17]:
months = df["time"].dt.strftime("%Y-%m")
test_months = months.unique()[-6:]
preds = []
for m in test_months:
    tr = df[months < m]
    te = df[months == m]
    model = Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=1.0))]).fit(tr[feats], tr["consumption_mwh"])
    preds.append(pd.Series(model.predict(te[feats]), index=te.index))
oos = pd.concat(preds)
by_month = pd.DataFrame({"month": months[oos.index], "err2": (y[oos.index] - oos) ** 2})
by_month.groupby("month")["err2"].mean().pow(0.5).round(0).rename("rmse")

month
2023-07     928.0
2023-08     921.0
2023-09     929.0
2023-10    1004.0
2023-11    1005.0
2023-12    1011.0
Name: rmse, dtype: float64

In [18]:
gs = GridSearchCV(Pipeline([("scale", StandardScaler()), ("ridge", Ridge())]),
                  param_grid={"ridge__alpha": [0.01, 1, 100]},
                  cv=TimeSeriesSplit(n_splits=4, gap=1), scoring="neg_root_mean_squared_error").fit(X, y)
pd.DataFrame(gs.cv_results_)[["param_ridge__alpha", "mean_test_score", "rank_test_score"]].round(1)

,param_ridge__alpha,mean_test_score,rank_test_score
0,0.0,-1014.5,3
1,1.0,-1014.4,2
2,100.0,-1014.1,1


The curve is flat across alpha: regularisation does not matter with 17k rows and 5 features.
Saying "alpha does not matter here" is a finding.

## Checklist: is my evaluation honest?

1. Training rows all precede test rows (print the fold boundaries).
2. `gap` at least the forecast horizon.
3. Scaler / imputer / encoder inside the pipeline, refit per fold.
4. Hyper-parameters chosen on folds that are not the final test period.
5. The metric is compared with a naive baseline on the same test rows.
6. Per-fold scores shown, not only the mean.

## Quick reference

| Task | Call |
|---|---|
| time-ordered folds | `TimeSeriesSplit(n_splits=5, gap=h, test_size=None)` |
| see the folds | `for tr, te in cv.split(X): print(tr, te)` |
| score with CV | `cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")` |
| several metrics | `cross_validate(..., scoring=[...], return_train_score=True)` |
| tune | `GridSearchCV(pipe, {"ridge__alpha": [...]}, cv=cv)`; `gs.cv_results_` |
| walk-forward | loop over blocks: fit on `iloc[:start]`, predict `iloc[start:start+n]` |